In [1]:
import pandas as pd
import lizard
import os
import csv

# Paths
PROCESSED_DIR = r"C:\Users\HP\Desktop\thesis_preprocessing\data\processed"

# We are using the newly generated V2 files
FILES =[
    "01_Original_12k_V2.csv",
    "02_Distractor_A_Swapped_12k_V2.csv",
    "03_Distractor_B_Shuffled_12k_V2.csv"
]

ext_map = {
    "java": ".java",
    "python": ".py",
    "go": ".go",
    "ruby": ".rb",
    "php": ".php",
    "javascript": ".js"
}

def calculate_adak_robust(row):
    # Force full string read
    code = str(row['original_code']).strip()
    comment = str(row['comment']).strip()
    lang = str(row['language']).lower()
    
    # 1. Accurate MCV (Method Comment Volume)
    comment_lines = len([line for line in comment.splitlines() if line.strip()])
    if comment_lines == 0: 
        comment_lines = 1 
    mcv = comment_lines * 0.8
    
    # 2. Accurate SFV (Source Functional Volume)
    ext = ext_map.get(lang, ".txt")
    dummy_filename = f"temp_file{ext}"
    
    try:
        analysis = lizard.analyze_file.analyze_source_code(dummy_filename, code)
        if analysis.function_list:
            token_count = analysis.function_list[0].token_count
        else:
            token_count = analysis.token_count
    except:
        token_count = 0
        
    # CRITICAL FALLBACK
    if token_count < 5:
        token_count = len(code.split())
        
    if token_count == 0:
        token_count = 1 
            
    sfv = max(token_count * 0.3, 0.3)
    
    # 3. Calculate Adak Index
    adak_index = ((100 * mcv) / sfv) - 100
    
    # CHANGED: Return a standard Python tuple to avoid Pandas dimension errors
    return mcv, sfv, adak_index

print("Setup Complete. Robust Adak function defined.")

Setup Complete. Robust Adak function defined.


In [2]:
print("Calculating Adak Index for 36,000 rows. Please wait...\n")

for file_name in FILES:
    print(f"Processing: {file_name}")
    file_path = os.path.join(PROCESSED_DIR, file_name)
    
    # Load dataset
    df_temp = pd.read_csv(file_path)
    
    # Apply calculation using the bulletproof zip unpacking method
    df_temp['mcv'], df_temp['sfv'], df_temp['adak_index'] = zip(*df_temp.apply(calculate_adak_robust, axis=1))
    
    # Save back to the file
    df_temp.to_csv(file_path, index=False, quoting=csv.QUOTE_ALL)
    print(f"✓ Saved {file_name}")

print("\nAll datasets successfully updated with MCV, SFV, and Adak Index.")

Calculating Adak Index for 36,000 rows. Please wait...

Processing: 01_Original_12k_V2.csv
✓ Saved 01_Original_12k_V2.csv
Processing: 02_Distractor_A_Swapped_12k_V2.csv
✓ Saved 02_Distractor_A_Swapped_12k_V2.csv
Processing: 03_Distractor_B_Shuffled_12k_V2.csv
✓ Saved 03_Distractor_B_Shuffled_12k_V2.csv

All datasets successfully updated with MCV, SFV, and Adak Index.


In [3]:
print("--- DATA INTEGRITY VALIDATION ---")

# Load the updated Original dataset to check the math
check_df = pd.read_csv(os.path.join(PROCESSED_DIR, FILES[0]))

# Group by language to check the Median SFV
stats = check_df.groupby('language')['sfv'].agg(['median', 'min', 'max']).round(2)
print(stats)

# Explicit PHP Check
php_median_sfv = stats.loc['php', 'median']
if php_median_sfv <= 1.0:
    print("\n[ALERT] Data is still corrupted. PHP Median SFV is too low.")
else:
    print("\n[VALIDATION PASSED] All languages show healthy code complexity (SFV). The PHP parser bug is fixed!")

--- DATA INTEGRITY VALIDATION ---
            median  min    max
language                      
go            15.0  2.4  138.0
java          17.7  2.4  151.8
javascript    15.9  1.5  151.2
php            9.9  2.1  115.8
python        20.4  2.7  148.5
ruby          13.8  1.5  147.0

[VALIDATION PASSED] All languages show healthy code complexity (SFV). The PHP parser bug is fixed!
